# Timpla — test a photo
Run the three cells in order. The last cell lets you upload a photo (maximum **5 MB**).

This uses the existing author's **YOLOv8m fruit and vegetable model**. It does not train a new model or connect the Expo app. CPU works; a GPU is optional. Photos are uploaded to your Colab runtime for testing.


In [ ]:
#@title 1. Set up
%pip install -q ultralytics==8.4.182 gdown==5.2.0 pillow-heif==1.1.1


In [ ]:
#@title 2. Load the existing model
import os
os.environ["YOLO_AUTOINSTALL"] = "False"

from pathlib import Path
import hashlib
import io
import json
import warnings
import gdown
import torch
from PIL import Image, ImageOps, UnidentifiedImageError
from pillow_heif import register_heif_opener
from IPython.display import display
from google.colab import files

register_heif_opener()
open_image = Image.open
from ultralytics import YOLO

MODEL = Path("/content/timpla-author-yolov8m.pt")
MODEL_SHA256 = "a18624d446e2fea08384949115de0b49446f0be2487fc954a37a9d962f71e5d3"
if not MODEL.exists():
    downloaded = gdown.download(
        id="1XALsPwM4850LkmqsXIfmGhPj8s_nhuiK",
        output=str(MODEL), quiet=False,
    )
    if not downloaded:
        raise RuntimeError("The model download is unavailable. Rerun this cell later.")

if MODEL.stat().st_size != 52_089_985:
    raise RuntimeError("Incomplete model download. Delete timpla-author-yolov8m.pt in Colab's Files panel, then rerun this cell.")
digest = hashlib.sha256()
with MODEL.open("rb") as stream:
    for block in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(block)
if digest.hexdigest() != MODEL_SHA256:
    raise RuntimeError("Model verification failed. Do not load this file.")

model = YOLO(str(MODEL), task="detect")
assert len(model.names) == 63, "Unexpected model classes."
allowed_classes = [i for i, name in model.names.items() if name not in {"Strawberry", "Tomato"}]
device = 0 if torch.cuda.is_available() else "cpu"
torch.set_num_threads(min(4, os.cpu_count() or 1))
print(f"Ready — {len(allowed_classes)} fruit and vegetable classes, using {device}.")
print("Run cell 3 to upload a photo.")


In [ ]:
#@title 3. Upload a photo and scan (run this cell again for another photo)
CONFIDENCE = 0.35
uploaded = files.upload()
output_dir = Path("/content/timpla-photo-results")
output_dir.mkdir(exist_ok=True)

for index, (filename, content) in enumerate(uploaded.items(), start=1):
    if not 0 < len(content) <= 5_000_000:
        print(f"{filename}: choose a non-empty image no larger than 5 MB.")
        continue
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error", Image.DecompressionBombWarning)
            with open_image(io.BytesIO(content)) as source:
                if source.format not in {"JPEG", "PNG", "WEBP", "BMP", "TIFF", "HEIF", "AVIF"}:
                    raise ValueError("Choose a supported image.")
                if source.width * source.height > 40_000_000:
                    raise ValueError("Choose an image no larger than 40 megapixels.")
                if getattr(source, "n_frames", 1) != 1:
                    raise ValueError("Choose a still image.")
                source.load()
                picture = ImageOps.exif_transpose(source).convert("RGB")
    except (UnidentifiedImageError, OSError, ValueError, Image.DecompressionBombWarning, Image.DecompressionBombError) as error:
        print(f"{filename}: invalid or unsupported image. {error}")
        continue

    result = model.predict(
        picture, imgsz=640, conf=CONFIDENCE, iou=0.5,
        classes=allowed_classes, device=device, verbose=False,
    )[0]
    ingredients = {}
    detections = []
    for box in result.boxes:
        class_id = int(box.cls.item())
        name = model.names[class_id].split("/")[0]
        confidence = float(box.conf.item())
        ingredients[name] = max(ingredients.get(name, 0), confidence)
        detections.append({
            "ingredient": name, "confidence": round(confidence, 4),
            "class_id": class_id,
            "bbox_xyxy": [round(float(value), 2) for value in box.xyxy[0].tolist()],
        })

    result.names = {i: name.split("/")[0] for i, name in result.names.items()}
    annotated = Image.fromarray(result.plot()[..., ::-1])
    print(filename)
    preview = annotated.copy()
    preview.thumbnail((900, 900))
    display(preview)
    for name, confidence in sorted(ingredients.items(), key=lambda item: item[1], reverse=True):
        print(f"• {name}: {confidence:.0%} confidence")
    if not ingredients:
        print("No ingredients detected. Try a clearer photo of fruit or vegetables.")
    annotated.save(output_dir / f"scan-{index}.jpg")
    (output_dir / f"scan-{index}.json").write_text(
        json.dumps({"image": filename, "detections": detections}, indent=2)
    )
    print("Confidence is the model's score, not measured accuracy.")
